In [1]:
import pandas as pd
import numpy as np

TARGETS = ["X"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3", 
    "LSG_1",
    "LSG_2"  
]

results = pd.read_excel("resultados.xlsx")


In [2]:
best_models_tables = {}

summary_all = []     # estatísticas com todos
summary_top10 = []   # estatísticas com top 10

N = 10

for target in TARGETS:
    for dataset in SETS:
        
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            table = results[
                ["model", "Neurons", col_r2, col_mse]
            ].sort_values(by=col_r2, ascending=False)
            
            # 🔹 Remove colchetes
            for col in [col_r2, col_mse]:
                table[col] = (
                    table[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )
            
            best_models_tables[f"{dataset}_{target}"] = table
            
            # ===============================
            # 🔹 Estatísticas - TODOS
            # ===============================
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })
            
            # ===============================
            # 🔹 Estatísticas - TOP 10
            # ===============================
            top10 = table.head(N)
            
            summary_top10.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": top10[col_r2].mean(),
                "std_r2": top10[col_r2].std(),
                "mean_mse": top10[col_mse].mean(),
                "std_mse": top10[col_mse].std()
            })


# 🔹 DataFrames finais
df_summary_all = pd.DataFrame(summary_all)
df_summary_top10 = pd.DataFrame(summary_top10)


# 🔹 Exportar para duas abas
with pd.ExcelWriter("Resumo_Estatisticas.xlsx") as writer:
    df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
    df_summary_top10.to_excel(writer, sheet_name="Top_10_Modelos", index=False)

print("Arquivo exportado com duas abas com sucesso.")


Arquivo exportado com duas abas com sucesso.


In [3]:
df_summary_top10

,dataset,target,mean_r2,std_r2,mean_mse,std_mse
0,Train_1,X,0.996370,0.000350,0.000390,0.000038
1,Train_2,X,0.997616,0.000213,0.000201,0.000018
2,Val,X,0.420014,0.162349,0.055264,0.015469
3,Test_1,X,0.987178,0.001769,0.001338,0.000185
4,Test_2,X,0.931521,0.019374,0.005626,0.001592
5,Test_3,X,0.299231,0.216149,0.064276,0.019826


In [4]:
best_only = []
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            idx_best = results[col_r2].idxmax()
            
            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": results.loc[idx_best, "model"],
                "Neurons": results.loc[idx_best, "Neurons"],
                "Best_R2": results.loc[idx_best, col_r2]
            })

best_only_df = pd.DataFrame(best_only)

print("\n===== MELHOR MODELO POR TARGET/DATASET =====")
print(best_only_df)



===== MELHOR MODELO POR TARGET/DATASET =====
  Target  Dataset                                   Best_Model     Neurons  \
0      X  Train_1     model_arch8-4_r0.01_Ld0.3_Lp0.7_seed8808      [8, 4]   
1      X  Train_2     model_arch8-4_r0.01_Ld0.3_Lp0.7_seed7182      [8, 4]   
2      X      Val     model_arch8-4_r0.01_Ld0.7_Lp0.3_seed8808      [8, 4]   
3      X   Test_1      model_arch8-4_r0.9_Ld0.7_Lp0.3_seed7914      [8, 4]   
4      X   Test_2  model_arch16-8-4_r0.01_Ld0.7_Lp0.3_seed7182  [16, 8, 4]   
5      X   Test_3  model_arch16-8-4_r0.01_Ld0.7_Lp0.3_seed5657  [16, 8, 4]   

    Best_R2  
0  0.997084  
1  0.997986  
2  0.731665  
3  0.991087  
4  0.963050  
5  0.826442  
